**GEOG5415M Programming for Spatial Data Science**

# Week 7: Machine Learning in Action

In this practical we will go through a worked example of machine learning in action. We are going to replicate some of the work in the paper:

 - Asher, M., Oswald, Y., & Malleson, N. (2025). Understanding pedestrian dynamics using machine learning with real-time urban sensors. _Environment and Planning B: Urban Analytics and City Science_, 52(8), 1994-2017. DOI:[10.1177/23998083251319058](https://doi.org/10.1177/23998083251319058)

The aim of the work is to build a model that can predict _pedestrian footfall_ for a particular hour, given information about the time, the weather, and the built environment.

If you are interested, the original code, in full, is available from the paper's Github repository: 

 - [github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis](https://github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis)

We will go through the following steps:
 1. Data preparation (including downloading, cleaning and linking)
 1. Data analysis (look for missing data and look at data distributions)
 1. Model the data (including model selection)

In [1]:
# import required packages

# XXXX 

import geopandas as gpd
import pandas as pd
import seaborn as sns
from scipy import stats
import numpy as np

import matplotlib.pyplot as plt

# import the required machine learning packages
from sklearn import cluster
from sklearn.preprocessing import scale

# set seaborn plotting theme to white
sns.set_theme(style="white")

# Data Preparation

We need to download, process, clean and merge the following data sources:

 - footfall counts (our target)
 - weather conditions
 - built environment features
 - date information (public holidays, school term times, etc,)

If you are interested, the full code is organised into various notebooks in the original repositoy's [PreparingData](https://github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis/1.%20PreparingData) directory.

## Downloading and reading data

### Footfall counts

The first dataset we need is the footfall counts -- i.e. the counts of pedestrians who pass Melbourne's footfall cameras every hour. In the paper, we had to find the data on the Melbourne Open Data Portal, download a few different files (one for old counts, one for new ones) and merge them. If you want to see the code to do this in full, have a look at the [Data Preparation Folder](https://github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis/1.%20PreparingData) in the paper's repository. For this practical, we have made the data available with the notebook to save some time, but there is still a lot of cleaning and preparation to do.

The data are stored in the file called "sensor_counts.csv.gz" in the data/week_7 directory. Note that the file has the '.gz' extension. This is short for 'gzip' and it means that the csv file has been compressed using an algorithm called 'gzip' so that it takes up less space. Fortunately pandas makes it really easy to read and write files using the argument `compression='gzip'`. As with other data, we can read it directly from the module's github repository

<font color='orchid'> <b>Run the code below to read the compressed csv file and assign it to a variable called `sensor_counts`</b></font>.

In [2]:
url_to_sensor_data = "https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_7/sensor_counts.csv.gz"
sensor_counts = pd.read_csv(url_to_sensor_data, compression='gzip')

Have a look at the sensor counts. Each row holds the number of counts from a sensor for a particular hour (the `hourly_counts`) column, as well as some other information. There are quite a lot of rows!

In [3]:
sensor_counts

,Unnamed: 0,datetime,year,month,mdate,day,time,sensor_id,hourly_counts
0,0,2019-11-01 17:00:00,2019,November,1,Friday,17,34,300
1,1,2019-11-01 17:00:00,2019,November,1,Friday,17,39,604
2,2,2019-11-01 17:00:00,2019,November,1,Friday,17,37,216
3,3,2019-11-01 17:00:00,2019,November,1,Friday,17,40,627
4,4,2019-11-01 17:00:00,2019,November,1,Friday,17,36,774
...,...,...,...,...,...,...,...,...,...
5480637,1850385,2024-03-22 15:00:00,2024,March,22,Friday,15,45,1388
5480638,1850388,2023-05-27 13:00:00,2023,May,27,Saturday,13,63,1052
5480639,1850389,2024-03-23 16:00:00,2024,March,23,Saturday,16,50,557
5480640,1850390,2024-04-17 16:00:00,2024,April,17,Wednesday,16,20,478


There is a column that we don't need so lets get rid of it. The easiest way to do this is to use the `sensor_counts.drop()` function. For example, if you wanted to remove a column called 'MyColumn' you could remove it with:
```python
sensor_counts.drop(colums = ['MyColumn'])
```

<font color='orchid'> <b>Edit the code below remove the column called 'Unnamed: 0'.</b></font>.

In [4]:
sensor_counts = sensor_counts.drop(columns = ['Unnamed: 0'])

The next chunk will check that the column removal worked correctly. If it runs without an error then you have done it right.
(Ask someone if you'd like us to explain how it works)

In [5]:
columns_still_in_df = [c for c in ['Unnamed: 0'] if c in sensor_counts.columns]
if columns_still_in_df:
    raise ValueError(f"These columns are still in the dataset: {columns_still_in_df}")
print("All columns removed successfully")

All columns removed successfully


### Sensor locations

We know what the counts at each sensor are, but we don't know _where_ the sensors are located. We need to know this so that we can map the sensor counts, and also so that we can join the sensors to other spatial data about the local built environment. The sensor locations are a separate file that we donloaded from the Melbourne Data Portal. For conveinence we have put the file on module repository so you can load it easily:

In [6]:
sensor_locations = pd.read_csv("https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_7/sensor_locations.csv")
sensor_locations

,Unnamed: 0,sensor_id,Name,sensor_name,installation_date,status,note,Latitude,Longitude,location,Note,Location_Type,Status
0,0,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.965210,"(-37.81573422, 144.96521044)",NaN,NaN,NaN
1,1,50,Faraday St-Lygon St (West),Lyg309_T,30/11/2017,A,NaN,-37.798082,144.967210,"(-37.79808191, 144.96721014)",NaN,NaN,NaN
2,2,73,Bourke St - Spencer St (South),Bou655_T,02/10/2020,I,NaN,-37.816957,144.954154,"(-37.81695684, 144.95415373)",NaN,NaN,NaN
3,3,66,State Library - New,QVN_T,06/04/2020,A,NaN,-37.810578,144.964443,"(-37.81057845, 144.96444294)",NaN,NaN,NaN
4,4,59,Building 80 RMIT,RMIT_T,13/02/2019,A,NaN,-37.808256,144.963049,"(-37.80825648, 144.96304859)",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,128,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.961860,"(-37.82590962, 144.96185972)",NaN,Outdoor,A
138,129,150,narrm ngarrgu Library - Level 1 Main Stairs B,narrLibL1MB_T,2023-10-23,NaN,NaN,-37.807912,144.958201,"(-37.80791198, 144.95820087)",NaN,Indoor,A
139,130,152,narrm ngarrgu Library - Level 2 - Study Area L...,narrLibL2S1_T,2023-10-23,NaN,NaN,-37.807767,144.958440,"(-37.80776728, 144.95843977)",NaN,Indoor,A
140,131,154,narrm ngarrgu Library - Level 3 Children's Lib...,narrLibL3C1_T,2023-10-23,NaN,NaN,-37.807784,144.958628,"(-37.80778437, 144.95862772)",NaN,Indoor,A


Again, you need to get rid of that strange column.
    
<font color='orchid'> <b>Edit the code below remove the column called 'Unnamed: 0'.</b></font>.

In [7]:
sensor_locations = sensor_locations.drop(columns = ['Unnamed: 0'])

### Merge the sensor counts with the sensor locations

Now that we have counts for the sensors as well as their locations, we can merge the two DatFrames. This is made each because both datasets have a column called `sensor_id`, which uniquely identifies a sensor. 

We will use the `pd.merge` function to merge them. Have a quick look at the [pandas.merge() documentation](https://pandas.pydata.org/docs/reference/api/pandas.merge.html). The function has two _requred) parameters: 'left' and 'right'. These are the two datasets that you want to merge. In our case it also needs the 'on' parameter so that it knows which column to use to merge the datasets. For example, if we wanted to merge two datasets, `df_a` and `df_b`, using column `id` as the linking column we would use `merge` like this:
```python
merged_df = pd.merge(df_a, df_b, on='id')
```
<font color='orchid'> <b>Edit the code below to create a new variable called `location_counts` by merging the `sensor_locations` and `sensor_counts` dataframes using the `sensor_id` column</b></font>.

In [8]:
location_counts = pd.merge(sensor_locations, sensor_counts, on='sensor_id')
location_counts

,sensor_id,Name,sensor_name,installation_date,status,note,Latitude,Longitude,location,Note,Location_Type,Status,datetime,year,month,mdate,day,time,hourly_counts
0,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,NaN,NaN,2011-01-01 00:00:00,2011,January,1,Saturday,0,744
1,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,NaN,NaN,2011-01-01 01:00:00,2011,January,1,Saturday,1,444
2,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,NaN,NaN,2011-01-01 02:00:00,2011,January,1,Saturday,2,263
3,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,NaN,NaN,2011-01-01 03:00:00,2011,January,1,Saturday,3,169
4,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,NaN,NaN,2011-01-01 04:00:00,2011,January,1,Saturday,4,79
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5480637,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,Outdoor,A,2024-04-02 15:00:00,2024,April,2,Tuesday,15,160
5480638,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,Outdoor,A,2024-09-03 12:00:00,2024,September,3,Tuesday,12,149
5480639,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,Outdoor,A,2024-05-23 05:00:00,2024,May,23,Thursday,5,3
5480640,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,Outdoor,A,2024-05-30 05:00:00,2024,May,30,Thursday,5,5


## Load the other datasets

XXXX HERE - load other features, transport, and special dates (ideally just read a pre-prepared file)

To make sure this worked, lets check that your new `location_counts` dataframe has the same number of rows as the original `sensor_counts` dataframe

In [9]:
if len(location_counts) == len(sensor_counts):
    print("Merged dataframe has the correct number of rows")
else:
    print("Merged dataframe length", len(location_counts), "doesn't match the original sensor counts length", len(sensor_counts))


Merged dataframe has the correct number of rows


### Some final cleaning steps

Here are some final steps that we need to do to prepare the data for anlaysis later. We didn't always know that these were necessary at first, so often we would do some anlaysis, find out something didn't work, and come back here to do some more data preparation.

The chunks below are mostly self explanatory, <font color='orchid'> <b>run the chunks below but check you know what they are doing.</b></font>.

In [10]:
# Repace the month names with month numbers (Jan -> 1, Feb -> 2, etc.)
months_dictionary = {'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June':6, 'July': 7, 'August': 8,
         'September': 9, 'October': 10, 'November': 11, 'December': 12}
location_counts['month'] = location_counts['month'].map(months_dictionary)

In [11]:
location_counts

,sensor_id,Name,sensor_name,installation_date,status,note,Latitude,Longitude,location,Note,Location_Type,Status,datetime,year,month,mdate,day,time,hourly_counts
0,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,NaN,NaN,2011-01-01 00:00:00,2011,1,1,Saturday,0,744
1,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,NaN,NaN,2011-01-01 01:00:00,2011,1,1,Saturday,1,444
2,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,NaN,NaN,2011-01-01 02:00:00,2011,1,1,Saturday,2,263
3,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,NaN,NaN,2011-01-01 03:00:00,2011,1,1,Saturday,3,169
4,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,NaN,NaN,2011-01-01 04:00:00,2011,1,1,Saturday,4,79
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5480637,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,Outdoor,A,2024-04-02 15:00:00,2024,4,2,Tuesday,15,160
5480638,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,Outdoor,A,2024-09-03 12:00:00,2024,9,3,Tuesday,12,149
5480639,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,Outdoor,A,2024-05-23 05:00:00,2024,5,23,Thursday,5,3
5480640,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,Outdoor,A,2024-05-30 05:00:00,2024,5,30,Thursday,5,5


# Data Analysis

# Modelling